In [29]:
import os

os.environ["LIBGL_ALWAYS_SOFTWARE"] = "1"

In [30]:
from pathlib import Path

import nibabel.funcs
import nibabel.nifti1
import nibabel.orientations

image_path = Path(
    "/scratch/imaging/ds005237/sub-NDARINVXP963GZ5/func/sub-NDARINVXP963GZ5_task-restPA_run-01_bold.nii.gz"
)
original_image = nibabel.nifti1.load(image_path)  # pyright: ignore[reportAttributeAccessIssue]

# The image will be reoriented to RAS so that the display frame is right-handed (the face
# is not mirrored). The slice axis must stay the third axis and keep its direction for
# SliceTiming to apply.
axis_codes = "".join(nibabel.orientations.aff2axcodes(original_image.affine))
assert axis_codes.endswith("S"), axis_codes
original_image.shape

(104, 104, 72, 488)

In [31]:
import json

import numpy as np

sidecar = json.loads(
    (image_path.parent / image_path.name.removesuffix(".gz").removesuffix(".nii"))
    .with_suffix(".json")
    .read_text()
)
slice_timing = np.asanyarray(sidecar["SliceTiming"])

In [32]:
import scipy.stats

slice_ranks = scipy.stats.rankdata(
    (slice_timing * 10000).astype(int), method="dense"
).astype(int)

In [33]:
frames_per_second = np.reciprocal(
    sidecar["RepetitionTime"] / np.unique(slice_ranks).size
)
frames_per_second

np.float64(11.25)

In [34]:
volume_count = round(60 / sidecar["RepetitionTime"])
volume_count

75

In [35]:
# Only read the volumes that are rendered, then reorient to RAS
image = nibabel.funcs.as_closest_canonical(original_image.slicer[..., :volume_count])
data = image.get_fdata(dtype=np.float32)
data.shape

(104, 104, 72, 75)

In [36]:
import nitransforms

derivatives_path = Path("/scratch/imaging/ds005237-fmriprep/sub-NDARINVXP963GZ5")

# fMRIPrep's boldref -> T1w transform maps T1w coordinates to boldref coordinates,
# so its inverse maps BOLD world coordinates to T1w world coordinates
coregistration_transform = nitransforms.linear.load(
    derivatives_path
    / "func/sub-NDARINVXP963GZ5_task-restPA_run-01_from-boldref_to-T1w_mode-image_desc-coreg_xfm.txt"
)
t1w_from_bold_world = np.linalg.inv(coregistration_transform.matrix)  # pyright: ignore[reportAttributeAccessIssue]

# Head motion estimates: each transform maps boldref coordinates to the coordinates of
# the same tissue in that volume, so applying it to the head moves the head with the data
motion_correction_transforms = nitransforms.linear.load(
    derivatives_path
    / "func/sub-NDARINVXP963GZ5_task-restPA_run-01_from-orig_to-boldref_mode-image_desc-hmc_xfm.txt"
)
if not isinstance(motion_correction_transforms, nitransforms.linear.LinearTransformsMapping):
    raise TypeError

# Everything is drawn in the BOLD voxel frame (in mm), so that the BOLD slices stay axis-aligned
zooms = np.array(image.header.get_zooms()[:3])
display_from_bold_voxel = np.diag([*zooms, 1.0])
display_from_bold_world = display_from_bold_voxel @ np.linalg.inv(image.affine)
display_from_t1w_world = display_from_bold_world @ np.linalg.inv(t1w_from_bold_world)

In [37]:
import nibabel.affines
import scipy.ndimage

t1w_image = nibabel.nifti1.load("t1w.nii.gz")  # pyright: ignore[reportAttributeAccessIssue]
t1w_data = t1w_image.get_fdata(dtype=np.float32)
display_from_t1w_voxel = display_from_t1w_world @ t1w_image.affine

# Resample the T1w onto a 1 mm grid that is aligned with the display frame
t1w_resolution = 1.0
corners = np.stack(
    np.meshgrid(*[[0, n - 1] for n in t1w_data.shape], indexing="ij"), axis=-1
).reshape(-1, 3)
corners = nibabel.affines.apply_affine(display_from_t1w_voxel, corners)
t1w_origin = np.floor(corners.min(axis=0))
t1w_shape = tuple(
    np.ceil((corners.max(axis=0) - t1w_origin) / t1w_resolution).astype(int) + 1
)
display_from_resampled = np.eye(4)
display_from_resampled[:3, :3] *= t1w_resolution
display_from_resampled[:3, 3] = t1w_origin
t1w_voxel_from_resampled = np.linalg.inv(display_from_t1w_voxel) @ display_from_resampled
t1w_resampled = scipy.ndimage.affine_transform(
    t1w_data,
    t1w_voxel_from_resampled[:3, :3],
    t1w_voxel_from_resampled[:3, 3],
    output_shape=t1w_shape,
    order=1,
)

# Smooth before extracting the skin surface
t1w_smooth = scipy.ndimage.gaussian_filter(t1w_resampled, 1.0)
head_threshold = 0.1 * np.percentile(t1w_resampled, 99)

In [38]:
import pyvista as pv

plot_voxel = np.array([52, 48, 21])  # in the voxel coordinates of the original image
volume_center = (
    nibabel.affines.apply_affine(
        np.linalg.inv(image.affine) @ original_image.affine, plot_voxel
    )
    * zooms
)

t1w_grid = pv.ImageData(
    dimensions=t1w_shape, spacing=(t1w_resolution,) * 3, origin=t1w_origin
)
t1w_grid.point_data["smooth"] = t1w_smooth.flatten(order="F")

head = (
    t1w_grid.contour(isosurfaces=[head_threshold])  # pyright: ignore[reportArgumentType]
    .smooth_taubin(n_iter=50, pass_band=0.05)
)
head.compute_normals(
    cell_normals=False,
    point_normals=True,
    auto_orient_normals=True,
    inplace=True
)

PolyData (0x780d2570a800)
  N Cells:    579660
  N Points:   289864
  N Strips:   0
  X Bounds:   1.157e+01, 1.943e+02
  Y Bounds:   -1.378e+00, 2.079e+02
  Z Bounds:   -8.639e+01, 1.528e+02
  N Arrays:   2

In [39]:
import matplotlib
import matplotlib.colors

# The slices are drawn with this colormap, and the whole frame uses its lowest color as
# the background, so that the noise outside the head blends in without a mask
colormap = matplotlib.colormaps["grey"]
background_color = matplotlib.colors.to_hex(colormap(0.0))
clim = np.percentile(data, [60, 99.5])

line_color = "#404040"
head_color = "#a0a0a0"
measuring_color = "#2171b5"
motion_color = "#7b2cbf" 

background_color, clim

('#000000', array([  663., 14072.]))

In [40]:
font_path = Path("/work/charite/thesis/defense/fonts/ChariteTextOffice-Regular.ttf")

In [41]:
import pandas as pd

# Framewise displacement from fMRIPrep: how much the head moved since the previous volume
confounds = pd.read_csv(
    derivatives_path
    / "func/sub-NDARINVXP963GZ5_task-restPA_run-01_desc-confounds_timeseries.tsv",
    sep="\t",
)
framewise_displacement = confounds["framewise_displacement"].to_numpy()[:volume_count]
framewise_displacement_threshold = 0.5  # mm
np.flatnonzero(framewise_displacement > framewise_displacement_threshold)

array([ 7, 37, 38, 39, 40, 64, 65, 66, 67, 68])

In [42]:
from collections.abc import Sequence

import imageio.v3
from scipy.spatial.transform import Rotation, Slerp
from tqdm.auto import tqdm

motion_scale = 1.0  # increase to exaggerate the head motion

# fMRIPrep estimates one rigid transform per volume. The slice views fill in a volume one
# slice group at a time, so the head should also move from the previous volume's pose to
# this volume's pose while the volume is being acquired, and arrive when the last slice
# group is drawn. Each estimate is therefore placed at the time of its volume's last slice
# group, and the transforms in between are interpolated (spherical interpolation for the
# rotation, linear for the translation)
motion_matrices = motion_correction_transforms.matrix
motion_times = (
    np.arange(len(motion_matrices)) * sidecar["RepetitionTime"] + slice_timing.max()
)
motion_rotations = Slerp(
    motion_times,
    Rotation.from_rotvec(
        Rotation.from_matrix(motion_matrices[:, :3, :3]).as_rotvec() * motion_scale
    ),
)
motion_translations = motion_matrices[:, :3, 3] * motion_scale


def get_head_motion(seconds: float) -> np.ndarray:
    """Head motion at a time point as a transform in the display frame."""
    # Before the first and after the last estimate, hold the nearest one
    seconds = np.clip(seconds, motion_times[0], motion_times[-1])
    motion = np.eye(4)
    motion[:3, :3] = motion_rotations(seconds).as_matrix()
    motion[:3, 3] = [
        np.interp(seconds, motion_times, motion_translations[:, i]) for i in range(3)
    ]
    return display_from_bold_world @ motion @ np.linalg.inv(display_from_bold_world)


# Three equal columns: 3D view, sagittal above coronal, transverse
plot_slices = (
    # normal, camera position, row, column, rows spanned
    # ("z", "xy", 0, 1, 2),  # transverse
    ("y", "xz", 0, 1, 2),  # coronal
    ("x", "yz", 0, 2, 2),  # sagittal
)

window_size = [3840, 2160]

scales: Sequence[float] = [
    0.675,
    0.95,
]

writer = imageio.get_writer(
    "raw.webm",
    format="FFMPEG",  # pyright: ignore[reportArgumentType]
    codec="vp9",
    fps=frames_per_second,
    output_params=[
        "-crf",
        "32",
        # Constant-quality mode
        "-b:v",
        "0",
        # Keyframe every two seconds
        "-g",
        str(round(2 * frames_per_second)),
        # Seek index at start of file
        "-cues_to_front",
        "1",
        # Use multi-threaded encoding
        "-row-mt",
        "1",
    ],
)

grid = pv.ImageData(
    dimensions=np.array(data.shape[:3]) + 1,
    spacing=zooms,
    origin=-0.5 * zooms,
)
fov_center = np.array(grid.center)
fov_size = (np.array(grid.dimensions) - 1) * zooms

# The head lies supine like in the scanner: the bore (z) runs horizontally with the
# top of the head on the right and the body leaving the view on the left, anterior is up
camera_focal_point = fov_center.copy()
camera_focal_point += np.array([0.0, 0.0, -30.0])
camera_direction = np.array([-1.0, 0.0, 0.0])
camera_position = camera_focal_point + 720.0 * camera_direction / np.linalg.norm(
    camera_direction
)
camera_up = (0.0, 1.0, 0.0)

current_data = np.zeros_like(data[..., 0])
cumulative_data = np.zeros_like(data[..., 0])

for volume_index in tqdm(range(data.shape[-1])):
    for rank in tqdm(np.sort(np.unique(slice_ranks)), leave=False):
        slices = slice_ranks == rank
        seconds = volume_index * sidecar["RepetitionTime"] + slice_timing[slices].min()
        moved_head = head.transform(get_head_motion(seconds), inplace=False)
        current_data *= 0.0
        current_data[..., slices] = 1.0

        cumulative_data[..., slices] = data[..., slices, volume_index]

        grid.cell_data["current"] = current_data.flatten(order="F")
        grid.cell_data["cumulative"] = cumulative_data.flatten(order="F")

        plotter = pv.Plotter(
            shape=(2, 3),
            groups=[(np.s_[:], 0)]
            + [
                (np.s_[row : row + row_span], column)
                for _, _, row, column, row_span in plot_slices
                if row_span > 1
            ],  # pyright: ignore[reportArgumentType]
            off_screen=True,
            window_size=window_size,
            border=False,
        )

        # 3D view of the T1w with the slices that are currently being measured
        plotter.subplot(0, 0)
        plotter.set_background(background_color)  # pyright: ignore[reportArgumentType]
        plotter.add_mesh(
            moved_head,
            color=head_color,
            pbr=True,
            metallic=0.05,
            roughness=0.55,
            smooth_shading=True,
            split_sharp_edges=False,
        )  # pyright: ignore[reportArgumentType]
        for k in np.flatnonzero(slices):
            slab = pv.Box(
                (
                    grid.bounds.x_min,
                    grid.bounds.x_max,
                    grid.bounds.y_min,
                    grid.bounds.y_max,
                    (k - 0.5) * zooms[2],
                    (k + 0.5) * zooms[2],
                )
            )
            # plotter.add_mesh(slab, color=measuring_color, opacity=0.12, lighting=False)  # pyright: ignore[reportArgumentType]
            # plotter.add_mesh(
            #     slab.extract_all_edges(),  # pyright: ignore[reportArgumentType]
            #     color=measuring_color,
            #     line_width=2,
            # )
            band = (
                moved_head.clip_box(slab.bounds, invert=False)  # pyright: ignore[reportArgumentType]
                .extract_surface(algorithm="dataset_surface")
                .compute_normals(auto_orient_normals=True)
            )
            if band.n_points:
                # Lift the band slightly off the skin so it doesn't flicker against it
                band.points += 0.3 * band.point_data["Normals"]
                plotter.add_mesh(
                    band,
                    color=measuring_color,
                    lighting=False,
                    opacity=0.35,
                )  # pyright: ignore[reportArgumentType]
        plotter.enable_anti_aliasing("ssaa")
        plotter.enable_ssao(radius=15, bias=0.01)  # pyright: ignore[reportCallIssue]
        plotter.camera.focal_point = camera_focal_point
        plotter.camera.position = camera_position
        plotter.camera.up = camera_up
        plotter.camera.view_angle = 30.0
        plotter.reset_camera_clipping_range()

        # Sagittal and coronal slices stacked in the middle, transverse on the right
        for i, (normal, camera_view, row, column, row_span) in enumerate(plot_slices):
            plotter.subplot(row, column)

            plotter.set_background(background_color)  # pyright: ignore[reportArgumentType]

            intersection = grid.slice(normal=normal, origin=volume_center)  # pyright: ignore[reportArgumentType]
            plotter.add_mesh(
                intersection,
                scalars="cumulative",
                cmap=colormap,
                clim=clim,
                lighting=False,
                show_edges=False,
                show_scalar_bar=False,
            )
            # current = intersection.threshold(0.5, scalars="current")
            # if current.n_cells:
            #     plotter.add_mesh(
            #         current,  # pyright: ignore[reportArgumentType]
            #         color=measuring_color,  # pyright: ignore[reportArgumentType]
            #         opacity=0.35,
            #         lighting=False,
            #         show_edges=False,
            #     )

            plotter.enable_anti_aliasing("ssaa")

            plotter.enable_parallel_projection()  # pyright: ignore[reportCallIssue]

            plotter.camera_position = camera_view  # pyright: ignore[reportAttributeAccessIssue]
            plotter.camera.focal_point = intersection.center

            sizes = np.delete(fov_size, "xyz".index(normal))
            aspect = (window_size[0] / 3) / (window_size[1] * row_span / 2)
            scale = max(sizes[1], sizes[0] / aspect) * 0.5 * 1.02
            plotter.camera.parallel_scale = scale * scales[i]

        plotter.subplot(0, 0)
        plotter.set_background(background_color)  # pyright: ignore[reportArgumentType]
        padding = 0.03
        plotter.add_text(
            f"{seconds:5.1f}s",
            position=(padding, padding),
            viewport=True,
            font_file=font_path.as_posix(),
            font_size=36,
            color="white",
        ).prop.justification_vertical = "bottom"  # pyright: ignore[reportAttributeAccessIssue]

        frame = plotter.screenshot(return_img=True)
        plotter.close()
        # imageio.v3.imwrite(f"frame-{volume_index:03d}-{rank}.png", frame)  # pyright: ignore[reportArgumentType, reportCallIssue]
        # break
        writer.append_data(frame)  # pyright: ignore[reportArgumentType]
    # break
writer.close()

  0%|          | 0/75 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

... (66 outputs left out)

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

In [43]:
camera_focal_point

array([103., 103.,  41.])